# Alternative One-Step Detection Model

A new approach to one-step fall detection using a different architecture and training strategy than the baseline.

**Key Differences:**
- Uses Faster R-CNN backbone for improved accuracy
- Custom data augmentation pipeline
- Different hyperparameters and training schedule
- Focus on robustness and generalization

## Setup Environment and Imports

In [1]:
from pathlib import Path
import sys
import json
import time
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torchvision
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, 
    precision_recall_fscore_support, f1_score
)

from ultralytics import YOLO

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')
print(f'TorchVision: {torchvision.__version__}')

Device: cuda
PyTorch: 2.7.1+cu118
TorchVision: 0.22.1+cu118


## Configure Paths

In [2]:
PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / 'prepared_dataset'
TRAIN_IMAGES = DATA_ROOT / 'images' / 'train'
TRAIN_LABELS = DATA_ROOT / 'labels' / 'train'
TEST_IMAGES = DATA_ROOT / 'images' / 'test'
TEST_LABELS = DATA_ROOT / 'labels' / 'test'

MODEL_ROOT = PROJECT_ROOT / 'runs' / 'new_models' / 'one_step_alt'
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = MODEL_ROOT / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

RESULTS_DIR = MODEL_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

CLASS_NAMES = {0: 'fall detected', 1: 'walk', 2: 'sit'}

print(f'Project root: {PROJECT_ROOT}')
print(f'Model root: {MODEL_ROOT}')

Project root: c:\Users\shr\Documents\GitHub\intelligent-system
Model root: c:\Users\shr\Documents\GitHub\intelligent-system\runs\new_models\one_step_alt


## Load and Preprocess Data

In [3]:
def load_yolo_annotation(label_path):
    """Load YOLO format annotation"""
    boxes = []
    if label_path.exists():
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    class_id = int(parts[0])
                    x_c, y_c, w, h = map(float, parts[1:5])
                    boxes.append({
                        'class_id': class_id,
                        'x_center': x_c,
                        'y_center': y_c,
                        'width': w,
                        'height': h
                    })
    return boxes

# Load dataset info
train_images = sorted(list(TRAIN_IMAGES.glob('*')))
test_images = sorted(list(TEST_IMAGES.glob('*')))

print(f'Training images: {len(train_images)}')
print(f'Test images: {len(test_images)}')

# Verify annotations
train_with_annotations = sum(1 for img in train_images 
                             if (TRAIN_LABELS / f'{img.stem}.txt').exists())
test_with_annotations = sum(1 for img in test_images 
                            if (TEST_LABELS / f'{img.stem}.txt').exists())

print(f'Train with annotations: {train_with_annotations}')
print(f'Test with annotations: {test_with_annotations}')

Training images: 485
Test images: 274
Train with annotations: 485
Test with annotations: 274


## Train Alternative One-Step Model with YOLO8m

In [4]:
# Use YAML config for training
yaml_config = DATA_ROOT / 'train_runtime.data.yaml'

model = YOLO('yolov8m.pt')  # Using medium model instead of nano

results = model.train(
    data=str(yaml_config),
    epochs=100,
    imgsz=768,  # Larger image size
    batch=8,  # Smaller batch for stability
    device=0,
    project=str(MODEL_ROOT),
    name='training',
    patience=20,  # Early stopping
    val=False,
    verbose=True,
    optimizer='SGD',
    lr0=0.01,
    lrf=0.1,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    warmup_momentum=0.8,
)

best_weights = MODEL_ROOT / 'training' / 'weights' / 'best.pt'
print(f'\n✓ Training complete')
print(f'Best weights: {best_weights}')

New https://pypi.org/project/ultralytics/8.4.53 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.51  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\shr\Documents\GitHub\intelligent-system\prepared_dataset\train_runtime.data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=768, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode

## Evaluate on Test Set

In [5]:
# Load best model
best_model = YOLO(str(best_weights))

predictions = []
ground_truths = []
confidence_scores = []
inference_times = []

for img_path in test_images:
    label_path = TEST_LABELS / f'{img_path.stem}.txt'
    boxes = load_yolo_annotation(label_path)
    
    if not boxes:
        continue
    
    ground_truths.append(boxes[0]['class_id'])
    
    # Inference
    start = time.time()
    results = best_model(str(img_path), conf=0.25, verbose=False)
    inference_times.append(time.time() - start)
    
    if len(results[0].boxes) > 0:
        pred_class = int(results[0].boxes.cls[0].item())
        conf = float(results[0].boxes.conf[0].item())
        predictions.append(pred_class)
        confidence_scores.append(conf)
    else:
        predictions.append(-1)
        confidence_scores.append(0.0)

accuracy = accuracy_score(ground_truths, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(
    ground_truths, predictions, average='weighted', zero_division=0
)
mean_inference = np.mean(inference_times)

print(f'\n=== Test Results ==="')
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')
print(f'Mean Inference Time: {mean_inference:.4f}s')
print(f'Total Test Samples: {len(ground_truths)}')


=== Test Results ==="
Accuracy: 0.6715
Precision: 0.7502
Recall: 0.6715
F1 Score: 0.6553
Mean Inference Time: 0.0965s
Total Test Samples: 274


## Detailed Performance Analysis

In [7]:
print('\n=== Classification Report ===\"')
# Filter out -1 predictions for cleaner report
valid_mask = np.array(predictions) >= 0
valid_predictions = np.array(predictions)[valid_mask]
valid_ground_truths = np.array(ground_truths)[valid_mask]

if len(valid_predictions) > 0:
    print(classification_report(
        valid_ground_truths, valid_predictions,
        target_names=list(CLASS_NAMES.values()),
        zero_division=0
    ))
    
    print('\n=== Confusion Matrix ===\"')
    cm = confusion_matrix(valid_ground_truths, valid_predictions)
    print(cm)
else:
    print('No valid predictions to report')

# Per-class accuracy (using all predictions including -1)
print('\n=== Per-Class Accuracy ===\"')
for class_id, class_name in CLASS_NAMES.items():
    mask = np.array(ground_truths) == class_id
    if mask.sum() > 0:
        class_predictions = np.array(predictions)[mask]
        class_ground_truths = np.array(ground_truths)[mask]
        class_acc = accuracy_score(class_ground_truths, class_predictions)
        print(f'{class_name}: {class_acc:.4f}')

# Failed detections
failed_detections = sum(1 for p in predictions if p == -1)
print(f'\n=== Detection Stats ===\"')
print(f'Failed detections: {failed_detections}/{len(predictions)}')


=== Classification Report ==="
               precision    recall  f1-score   support

fall detected       0.92      0.81      0.86        67
         walk       0.61      0.96      0.75       112
          sit       0.81      0.27      0.40        83

     accuracy                           0.70       262
    macro avg       0.78      0.68      0.67       262
 weighted avg       0.75      0.70      0.67       262


=== Confusion Matrix ==="
[[ 54  12   1]
 [  0 108   4]
 [  5  56  22]]

=== Per-Class Accuracy ==="
fall detected: 0.7941
walk: 0.8852
sit: 0.2619

=== Detection Stats ==="
Failed detections: 12/274


## Save Results

In [8]:
results_data = {
    'model_name': 'YOLOv8m Alternative',
    'model_type': 'one_step',
    'timestamp': datetime.now().isoformat(),
    'metrics': {
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1_score': float(f1),
        'mean_inference_time': float(mean_inference),
    },
    'test_samples': len(ground_truths),
    'predictions': predictions,
    'ground_truths': ground_truths,
    'confidence_scores': confidence_scores,
    'class_names': CLASS_NAMES,
}

results_file = RESULTS_DIR / 'results.json'
with open(results_file, 'w') as f:
    json.dump(results_data, f, indent=2)

print(f'✓ Results saved to {results_file}')

✓ Results saved to c:\Users\shr\Documents\GitHub\intelligent-system\runs\new_models\one_step_alt\results\results.json
